In [ ]:
import torchimport torch.nn as nnimport torch.optim as optimfrom torchvision import datasets, transforms, modelsfrom torch.utils.data import DataLoader, Subsetimport osimport randomimport matplotlib.pyplot as pltimport numpy as npfrom tqdm import tqdmfrom collections import Counterfrom copy import deepcopyfrom sklearn.model_selection import train_test_splitfrom sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, roc_auc_scorefrom sklearn.preprocessing import label_binarizeimport cv2DATA_DIR = '/kaggle/input/datasets/joebeachcapital/realwaste/realwaste-main/RealWaste/'if not os.path.exists(DATA_DIR):    print(f"Error: Dataset directory not found at {DATA_DIR}. Please ensure the 'realwaste' dataset is added to your Kaggle notebook.")BATCH_SIZE = 32NUM_EPOCHS = 50LEARNING_RATE = 0.001PATIENCE = 5MODEL_SAVE_PATH = 'best_densenet121_waste_classifier_with_attention.pth'SEED = 42random.seed(SEED)np.random.seed(SEED)torch.manual_seed(SEED)torch.cuda.manual_seed_all(SEED)torch.backends.cudnn.deterministic = Truetorch.backends.cudnn.benchmark = Falseif torch.cuda.is_available():    try:        test_tensor = torch.zeros(1).to("cuda")        test_tensor.cpu()        device = torch.device("cuda")    except Exception as e:        device = torch.device("cpu")        print(f"CUDA is not usable on this environment ({e}). Falling back to CPU.")else:    device = torch.device("cpu")    print("CUDA is not available. Using CPU.")print(f"Using device: {device}")print(f"Random seed fixed: {SEED}")data_transforms = {    'train': transforms.Compose([        transforms.RandomResizedCrop(224),        transforms.RandomHorizontalFlip(),        transforms.ToTensor(),        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])    ]),    'val_test': transforms.Compose([        transforms.Resize(256),        transforms.CenterCrop(224),        transforms.ToTensor(),        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])    ]),}print("Loading full dataset for stratified splitting...")full_dataset_for_split = datasets.ImageFolder(DATA_DIR)class_names = full_dataset_for_split.classesnum_classes = len(class_names)print(f"Found {num_classes} classes: {class_names}")labels = np.array(full_dataset_for_split.targets)indices = np.arange(len(full_dataset_for_split))train_idx, temp_idx = train_test_split(indices, test_size=0.3, stratify=labels, random_state=SEED)val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, stratify=labels[temp_idx], random_state=SEED)train_idx, val_idx, test_idx = sorted(train_idx), sorted(val_idx), sorted(test_idx)train_dataset = deepcopy(full_dataset_for_split)train_dataset.transform = data_transforms['train']train_dataset = Subset(train_dataset, train_idx)val_dataset = deepcopy(full_dataset_for_split)val_dataset.transform = data_transforms['val_test']val_dataset = Subset(val_dataset, val_idx)test_dataset = deepcopy(full_dataset_for_split)test_dataset.transform = data_transforms['val_test']test_dataset = Subset(test_dataset, test_idx)image_datasets = {    'train': train_dataset,    'val': val_dataset,    'test': test_dataset}dataloaders = {    'train': DataLoader(image_datasets['train'], batch_size=BATCH_SIZE, shuffle=True, num_workers=2),    'val': DataLoader(image_datasets['val'], batch_size=BATCH_SIZE, shuffle=False, num_workers=2),    'test': DataLoader(image_datasets['test'], batch_size=BATCH_SIZE, shuffle=False, num_workers=2)}print(f"Train images: {len(image_datasets['train'])}")print(f"Validation images: {len(image_datasets['val'])}")print(f"Test images: {len(image_datasets['test'])}")print("\nGenerating data distribution chart...")def plot_data_distribution(datasets, class_names):    fig, axes = plt.subplots(1, 3, figsize=(18, 6))    fig.suptitle('Stratified Image Distribution Across Classes in Train, Validation, and Test Sets', fontsize=16)    for i, (phase, dataset) in enumerate(datasets.items()):        if isinstance(dataset, torch.utils.data.Subset):            targets = [dataset.dataset.targets[idx] for idx in dataset.indices]            class_counts = Counter(targets)        else:            class_counts = Counter(dataset.targets)                labels = [class_names[j] for j in sorted(class_counts.keys())]        counts = [class_counts[j] for j in sorted(class_counts.keys())]        axes[i].bar(labels, counts, color='skyblue')        axes[i].set_title(f'{phase.capitalize()} Set')        axes[i].set_xlabel('Waste Type')        axes[i].set_ylabel('Number of Images')        axes[i].tick_params(axis='x', rotation=45)        for idx, count in enumerate(counts):            axes[i].text(idx, count, str(count), ha='center', va='bottom')    plt.tight_layout(rect=[0, 0.03, 1, 0.95])    plt.show()plot_data_distribution(image_datasets, class_names)class ChannelAttention(nn.Module):    def __init__(self, channel, reduction=16):        super(ChannelAttention, self).__init__()        self.avg_pool = nn.AdaptiveAvgPool2d(1)        self.max_pool = nn.AdaptiveMaxPool2d(1)        self.fc = nn.Sequential(            nn.Linear(channel * 2, channel // reduction, bias=False),            nn.ReLU(inplace=True),            nn.Linear(channel // reduction, channel, bias=False),            nn.Sigmoid()        )    def forward(self, x):        avg_out = self.avg_pool(x)        max_out = self.max_pool(x)        out = torch.cat([avg_out, max_out], dim=1)        out = self.fc(out.view(x.size(0), -1)).view(x.size(0), x.size(1), 1, 1)        return x * out.expand_as(x)class DenseNet121WithAttention(nn.Module):    def __init__(self, num_classes):        super(DenseNet121WithAttention, self).__init__()        densenet121 = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)                self.features = densenet121.features        for param in self.features.parameters():            param.requires_grad = False                num_channels = 1024                self.attention = ChannelAttention(num_channels)        self.classifier = nn.Linear(num_channels * 7 * 7, num_classes)        def forward(self, x):        x = self.features(x)        x = self.attention(x)        x = x.view(x.size(0), -1)        x = self.classifier(x)        return xprint("Initializing DenseNet121 model with Attention...")model = DenseNet121WithAttention(num_classes).to(device)print("Calculating class weights for imbalanced data...")train_targets = [train_dataset.dataset.targets[idx] for idx in train_dataset.indices]class_counts = Counter(train_targets)total_samples = len(train_targets)class_weights = [total_samples / (num_classes * class_counts[i]) for i in range(num_classes)]class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)print(f"Calculated class weights: {class_weights}")criterion = nn.CrossEntropyLoss(weight=class_weights)optimizer = optim.Adam(list(model.attention.parameters()) + list(model.classifier.parameters()), lr=LEARNING_RATE)scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2)def train_model(model, criterion, optimizer, scheduler, num_epochs=NUM_EPOCHS, patience=PATIENCE):    best_acc = 0.0    best_val_loss = float('inf')    patience_counter = 0    train_losses = []    train_accuracies = []    val_losses = []    val_accuracies = []    for epoch in range(num_epochs):        print(f'Epoch {epoch+1}/{num_epochs}')        print('-' * 10)        for phase in ['train', 'val']:            if phase == 'train':                model.train()            else:                model.eval()            running_loss = 0.0            running_corrects = 0            for inputs, labels in tqdm(dataloaders[phase], desc=f"{phase.capitalize()} Epoch {epoch+1}"):                inputs = inputs.to(device)                labels = labels.to(device)                optimizer.zero_grad()                with torch.set_grad_enabled(phase == 'train'):                    outputs = model(inputs)                    _, preds = torch.max(outputs, 1)                    loss = criterion(outputs, labels)                    if phase == 'train':                        loss.backward()                        optimizer.step()                running_loss += loss.item() * inputs.size(0)                running_corrects += torch.sum(preds == labels.data)            epoch_loss = running_loss / len(image_datasets[phase])            epoch_acc = running_corrects.double() / len(image_datasets[phase])            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')            if phase == 'train':                train_losses.append(epoch_loss)                train_accuracies.append(epoch_acc.item())            else:                val_epoch_loss = epoch_loss                val_losses.append(epoch_loss)                val_accuracies.append(epoch_acc.item())                if epoch_acc > best_acc:                    best_acc = epoch_acc                    torch.save(model.state_dict(), MODEL_SAVE_PATH)                    print(f"Saved best model with accuracy: {best_acc:.4f}")                if epoch_loss < best_val_loss:                    best_val_loss = epoch_loss                    patience_counter = 0                else:                    patience_counter += 1                scheduler.step(val_epoch_loss)                if patience_counter >= patience:            print(f"Early stopping triggered after {patience_counter} epochs without improvement in validation loss.")            break    print(f'Training complete. Best validation accuracy: {best_acc:.4f}')    return model, train_losses, train_accuracies, val_losses, val_accuraciesprint("Starting training...")model_ft, train_losses, train_accuracies, val_losses, val_accuracies = train_model(model, criterion, optimizer, scheduler, NUM_EPOCHS, PATIENCE)print("Plotting training history...")plt.figure(figsize=(12, 5))plt.subplot(1, 2, 1)plt.plot(train_losses, label='Train Loss')plt.plot(val_losses, label='Validation Loss')plt.title('Loss per Epoch')plt.xlabel('Epoch')plt.ylabel('Loss')plt.legend()plt.grid(True)plt.subplot(1, 2, 2)plt.plot(train_accuracies, label='Train Accuracy')plt.plot(val_accuracies, label='Validation Accuracy')plt.title('Accuracy per Epoch')plt.xlabel('Epoch')plt.ylabel('Accuracy')plt.legend()plt.grid(True)plt.tight_layout()plt.show()print("\nEvaluating model on the test set...")model_ft.load_state_dict(torch.load(MODEL_SAVE_PATH))model_ft.eval()running_corrects = 0total_samples = 0all_preds = []all_labels = []all_prob = []with torch.no_grad():    for inputs, labels in tqdm(dataloaders['test'], desc="Testing"):        inputs = inputs.to(device)        labels = labels.to(device)        outputs = model_ft(inputs)        probs = torch.softmax(outputs, dim=1)        _, preds = torch.max(outputs, 1)        running_corrects += torch.sum(preds == labels.data)        total_samples += labels.size(0)        all_preds.extend(preds.cpu().numpy())        all_labels.extend(labels.cpu().numpy())        all_prob.append(probs.cpu().numpy())all_prob = np.concatenate(all_prob, axis=0)test_accuracy = running_corrects.double() / total_samplesprint(f'Test Accuracy: {test_accuracy:.4f}')print("\nClassification Report:")print(classification_report(all_labels, all_preds, target_names=class_names))all_labels_binarized = label_binarize(all_labels, classes=np.arange(num_classes))fpr = dict()tpr = dict()roc_auc = dict()for i in range(num_classes):    fpr[i], tpr[i], _ = roc_curve(all_labels_binarized[:, i], all_prob[:, i])    roc_auc[i] = auc(fpr[i], tpr[i])macro_roc_auc = roc_auc_score(all_labels_binarized, all_prob, average='macro')print(f'\nMacro ROC-AUC: {macro_roc_auc:.4f}')plt.figure(figsize=(10, 8))for i in range(num_classes):    plt.plot(fpr[i], tpr[i], label=f'{class_names[i]} (AUC = {roc_auc[i]:.2f})')plt.plot([0, 1], [0, 1], '--', color='gray', linewidth=1)plt.xlim([0, 1])plt.ylim([0, 1.05])plt.xlabel('False Positive Rate')plt.ylabel('True Positive Rate')plt.title('ROC Curve')plt.legend(loc='lower right')plt.grid(True)plt.show()cm = confusion_matrix(all_labels, all_preds)plt.figure(figsize=(10, 8))plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)plt.title('Confusion Matrix (DenseNet121 + Channel Attention)')plt.colorbar()tick_marks = np.arange(num_classes)plt.xticks(tick_marks, class_names, rotation=45, ha='right')plt.yticks(tick_marks, class_names)plt.tight_layout()plt.ylabel('True label')plt.xlabel('Predicted label')thresh = cm.max() / 2.for i in range(cm.shape[0]):    for j in range(cm.shape[1]):        plt.text(j, i, format(cm[i, j], 'd'),                 ha="center", va="center",                 color="white" if cm[i, j] > thresh else "black")plt.show()print("\nGenerating Grad-CAM visualizations for a few test images...")def generate_gradcam(model, input_image, target_layer, device):    model.eval()    input_image = input_image.unsqueeze(0).to(device)    for param in model.features.parameters():        param.requires_grad = True    activations = []    gradients = []    def forward_hook(module, input, output):        output.retain_grad()        activations.append(output)    def backward_hook(module, grad_input, grad_output):        gradients.append(grad_output[0].detach())    forward_hook_handle = target_layer.register_forward_hook(forward_hook)    backward_hook_handle = target_layer.register_full_backward_hook(backward_hook)    model.zero_grad()    output = model(input_image)    _, predicted_class = torch.max(output, 1)    score = output[:, predicted_class.item()]    score.backward(retain_graph=True)    forward_hook_handle.remove()    backward_hook_handle.remove()    for param in model.features.parameters():        param.requires_grad = False    activations_tensor = activations[0]    gradients_tensor = gradients[0]    weights = torch.mean(gradients_tensor, dim=(2, 3), keepdim=True)    heatmap = torch.relu((weights * activations_tensor).sum(dim=1)).squeeze()    heatmap = cv2.resize(heatmap.detach().cpu().numpy(), (input_image.shape[3], input_image.shape[2]))    heatmap = np.uint8(255 * heatmap)    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)    input_image_np = input_image.squeeze(0).cpu().numpy().transpose(1, 2, 0)    input_image_np = (input_image_np - input_image_np.min()) / (input_image_np.max() - input_image_np.min())    superimposed_img = cv2.addWeighted(np.uint8(input_image_np * 255), 0.6, heatmap, 0.4, 0)    superimposed_img_rgb = cv2.cvtColor(superimposed_img, cv2.COLOR_BGR2RGB)    return superimposed_img_rgb, predicted_class.item()num_samples = 4test_subset = Subset(image_datasets['test'], range(num_samples))test_loader_gradcam = DataLoader(test_subset, batch_size=1, shuffle=False)target_layer = model_ft.attentionfig, axes = plt.subplots(num_samples, 2, figsize=(10, num_samples * 5))fig.suptitle('Grad-CAM Visualizations (post-attention)', fontsize=16)test_dataset_orig = datasets.ImageFolder(DATA_DIR, transforms.ToTensor())test_subset_orig = Subset(test_dataset_orig, test_dataset.indices[:num_samples])test_loader_orig = DataLoader(test_subset_orig, batch_size=1, shuffle=False)for i, ((original_image, _), (input_tensor, _)) in enumerate(zip(test_loader_orig, test_loader_gradcam)):    gradcam_image, predicted_class_idx = generate_gradcam(model_ft, input_tensor.squeeze(0), target_layer, device)    axes[i, 0].imshow(np.transpose(original_image.squeeze(0), (1, 2, 0)))    true_class_idx = full_dataset_for_split.targets[test_dataset.indices[i]]    axes[i, 0].set_title(f'Original Image\nTrue Class: {class_names[true_class_idx]}')    axes[i, 0].axis('off')    axes[i, 1].imshow(gradcam_image)    axes[i, 1].set_title(f'Grad-CAM Heatmap\nPredicted Class: {class_names[predicted_class_idx]}')    axes[i, 1].axis('off')plt.tight_layout(rect=[0, 0.03, 1, 0.95])plt.show()print("\nScript finished.")